# 7단계 | 대표 P/N 실제 호출 결과 통합

- 기존 Vishay 응답과 검증 결과를 그대로 사용함
- 저장된 KYOCERA AVX·Panasonic 호출 기록으로 오프라인 실행함
- Panasonic 재호출 1회와 HTTP 503 이력을 별도 보존함
- 규격 응답이 없어 Panasonic 6개 항목 미평가로 유지함

In [1]:
from collections import Counter
from copy import deepcopy
from pathlib import Path
import hashlib
import json
import os

import matplotlib.pyplot as plt
from matplotlib import font_manager
import numpy as np
import pandas as pd
import pymupdf
import torch
from PIL import Image
from IPython.display import display

from table_detection import load_table_detector, load_structure_recognizer
from datasheet_analysis import (
    SPEC_FIELDS, detect_page_regions, extract_document_source, recognize_region_structure,
    select_extraction_evidence, validate_saved_extraction,
)

ROOT = Path.cwd()
RESULT_ROOT = ROOT
json_rel = Path('results/07_llm_extraction.json')
summary_rel = Path('results/07_llm_extraction_summary.csv')
errors_rel = Path('results/07_llm_extraction_errors.csv')
status_rel = Path('outputs/07_llm_spec_status.png')
error_chart_rel = Path('outputs/07_llm_extraction_errors.png')

saved = json.loads((ROOT / json_rel).read_bytes())
run = saved['remaining_pn_run']
retry = saved['panasonic_retry_run']
retry_record = retry['execution_record']
assert retry['requests_dispatched'] == retry_record['requests_dispatched'] == 1
assert retry['sdk_retry_attempts'] == 1 and retry['notebook_additional_requests'] == 0
assert retry_record['part_number'] == '4TPB68M'
assert retry_record['http_status'] == 503 and retry_record['raw_response_text'] is None
assert retry['specifications_pending'] == len(SPEC_FIELDS)
assert retry['specification_evaluation'] == 'not_evaluated'
print({k: retry_record[k] for k in ('part_number', 'requested_model', 'http_status',
                                    'processing_seconds', 'input_tokens', 'output_tokens', 'error_type')})
history = {k: v for k, v in saved.items() if k not in ('remaining_pn_run', 'panasonic_retry_run')}
vishay = history['independent_validation']
assert vishay['verification_counts'] == {'verified': 3, 'failed': 3, 'unverifiable': 0}
assert hashlib.sha256(history['responses'][0]['raw_response_text'].encode()).hexdigest() == vishay['raw_response_sha256']
records = run['execution_records']
assert [r['part_number'] for r in records] == ['TCJU337M016R0050E', '4TPB68M']
assert all(r['requests_dispatched'] == 1 for r in records)
assert run['requests_dispatched'] == sum(r['requests_dispatched'] for r in records) == 2
assert run['free_tier']['project_confirmed_by_user'] and run['sdk_retry_attempts'] == 1
for record in records:
    print({k: record[k] for k in ('part_number', 'requested_model', 'http_status', 'processing_seconds',
                                  'input_tokens', 'output_tokens', 'error_type')})
print('저장된 호출 이력 재사용함. Notebook 추가 요청 0건.')

{'part_number': '4TPB68M', 'requested_model': 'gemini-3.5-flash-lite', 'http_status': 503, 'processing_seconds': 3.2469713000027696, 'input_tokens': None, 'output_tokens': None, 'error_type': 'ServerError'}
{'part_number': 'TCJU337M016R0050E', 'requested_model': 'gemini-3.5-flash-lite', 'http_status': 200, 'processing_seconds': 5.803543299989542, 'input_tokens': 4787, 'output_tokens': 1036, 'error_type': None}
{'part_number': '4TPB68M', 'requested_model': 'gemini-3.5-flash-lite', 'http_status': 503, 'processing_seconds': 1.2667055999918375, 'input_tokens': None, 'output_tokens': None, 'error_type': 'ServerError'}
저장된 호출 이력 재사용함. Notebook 추가 요청 0건.


## 개발용 PDF 근거와 구조 검사

- 전송한 원문·페이지·좌표를 6단계 통합 결과와 개발 PDF에서 확인함

In [2]:
bundle = json.loads((ROOT / 'results/06_llm_input_samples.json').read_bytes())
documents = {r['document_id']: r for r in bundle['documents']}
paths = {}
splits = pd.read_csv(ROOT / 'data/dataset_split.csv', dtype=str).set_index('document_id')
part_numbers = pd.read_csv(ROOT / 'data/part_numbers.csv', dtype=str)
plans = {plan['document_id']: plan for plan in run['request_plans']}
for record in records:
    doc_id, target = record['document_id'], record['part_number']
    document, plan = documents[doc_id], plans[doc_id]
    assert document['split'] == splits.loc[doc_id, 'split'] == 'dev'
    assert len(part_numbers.loc[(part_numbers.document_id == doc_id) &
                                (part_numbers.part_number == target) & (part_numbers.split == 'dev')]) == 1
    paths[doc_id] = ROOT / 'data/datasheets' / document['filename']
    assert hashlib.sha256(paths[doc_id].read_bytes()).hexdigest() == document['pdf_sha256'] == plan['pdf_sha256']
    supplemental = (extract_document_source(paths[doc_id], doc_id, [10, 11, 12], [])[0]
                    if doc_id == 'kyocera_avx_tcj' else None)
    prepared = select_extraction_evidence(bundle, doc_id, target, supplemental)
    assert prepared['request']['evidence'] == plan['evidence']
    if target == retry_record['part_number']:
        assert plan == retry['request_plan']
        assert prepared['request']['occurrences'][0]['geometry_status'] == 'fallback_source_only'
    assert sorted({e['pdf_page'] for e in prepared['evidence']}) == plan['pdf_pages']
    with pymupdf.open(paths[doc_id]) as pdf:
        for entry in prepared['evidence']:
            number = entry['pdf_page']
            raw = pdf[number - 1].get_text('words')
            stored = prepared['pages'][number]
            ids = entry['word_ids']
            assert set(ids).isdisjoint(stored['hidden_word_ids'])
            assert all([raw[i][4], *raw[i][:4], *raw[i][5:8]] == stored['words'][i] for i in ids)
            assert entry['text'] == ' '.join(raw[i][4] for i in ids)
    if record['http_status'] == 200:
        replay = validate_saved_extraction(record['raw_response_text'], prepared)
        assert replay['structure']['status'] == 'passed'
        assert replay == record['independent_validation']
        checked = deepcopy(replay)
    else:
        assert record['http_status'] == 503 and record['raw_response_text'] is None
        assert 'independent_validation' not in record
paths['vishay_t55'] = ROOT / 'data/datasheets' / documents['vishay_t55']['filename']
print('개발 분할, P/N 등록, PDF 해시, 전송 근거와 저장 응답 검증함.')

개발 분할, P/N 등록, PDF 해시, 전송 근거와 저장 응답 검증함.


## KYOCERA AVX 규격별 원문 대조

- 자동 판정과 표·주문 코드의 원문 대조를 분리함
- Panasonic은 HTTP 503으로 규격 응답 없음

In [3]:
# KYOCERA AVX PDF의 실제 표와 주문 코드를 대조함
kyocera, panasonic = records
source_entries = {e['label']: e['text'] for e in plans[kyocera['document_id']]['evidence']
                  if e['pdf_page'] in (1, 6, 10)}
assert source_entries['exact_target_pn_row'] == 'TCJU337M016R0050E U 330 16 125 528 12 50 2800 2000 1300 700 1 3'
header = source_entries['table_header']
assert 'Capacitance (μF) Rated Voltage (V) Maximum Operating Temperature (ºC)' in header
assert 'ESR Max. @ 100kHz (mΩ)' in header
assert 'M = ±20%' in source_entries['ordering_guide']
assert 'ambient temperature of +25°C' in source_entries['series_technical_context']
assert 'CATEGORY 0, 1 (TEMPERATURE RANGE -55°C TO +125°C)' in source_entries['category_heading']
assert 'Apply 1.3x 2/3x rated voltage (Ur) at 125ºC' in source_entries['surge_test_conditions']

reviews = {
    'rated_voltage': ([6], '6쪽 대상 P/N 행의 정격전압 열에 16 V가 기재됨.'),
    'surge_voltage': ([6, 10], '6쪽 대상 행은 카테고리 1임. 10쪽에 125 °C에서의 시험 배율과 조건이 있으나 P/N별 서지전압 수치는 없음.'),
    'capacitance': ([6], '6쪽 대상 P/N 행과 정전용량 열의 330 μF가 일치함.'),
    'tolerance': ([1, 6], '1쪽 주문 코드 M은 ±20%임. 6쪽 대상 P/N의 허용오차 코드 M 및 시리즈 공통 규격과 일치함.'),
    'esr': ([1, 6], '6쪽 대상 P/N 행의 최대 ESR은 50 mΩ이고 헤더에 100 kHz가 표시됨. 1쪽 공통 기술 데이터 기준 +25 °C는 모델 응답 조건에 누락됨.'),
    'operating_temperature': ([6, 10], '6쪽 대상 P/N 최대 동작 온도는 +125 °C임. 카테고리 1 범위는 10쪽의 −55 °C~+125 °C이며 모델 응답은 상한만 제공함.'),
}
scope_issue = 'pn_field_or_series_scope_not_proved_by_automatic_check'
for field, (pages, note) in reviews.items():
    row = checked['specifications'][field]
    row['automated_verification_status'] = row['verification_status']
    row['automated_issues'] = row['issues'].copy()
    row['pdf_review'] = {'pages': pages, 'finding': note}

for field, expected in {
    'rated_voltage': ('16', 'V', 'pn', 6),
    'capacitance': ('330', 'μF', 'pn', 6),
    'tolerance': ('±20', '%', 'series', 1),
}.items():
    row = checked['specifications'][field]
    assert tuple(row['raw'][k] for k in ('value', 'unit', 'scope', 'page')) == expected
    assert row['schema_status'] == 'passed' and row['source_text_matched'] and row['issues'] == [scope_issue]
    row.update(verification_status='verified', issues=[])

esr = checked['specifications']['esr']
assert esr['source_text_matched'] and esr['issues'] == [scope_issue]
assert (esr['raw']['value'], esr['raw']['unit'], esr['raw']['condition']) == ('50', 'mΩ', '100kHz')
esr.update(verification_status='failed', issues=['esr_ambient_temperature_missing'])
temperature = checked['specifications']['operating_temperature']
assert temperature['source_text_matched'] and temperature['raw']['value'] == '125'
assert temperature['raw']['value_min'] is None and temperature['raw']['value_max'] is None
temperature['issues'].append('operating_temperature_range_missing')
assert checked['specifications']['surge_voltage']['raw']['status'] == 'not_found'

states = ('verified', 'failed', 'unverifiable')
extraction_states = ('found', 'not_found', 'ambiguous', 'extraction_error')
sets = [
    ('Vishay', vishay['document_id'], vishay['part_number'], vishay['specifications']),
    ('KYOCERA AVX', kyocera['document_id'], kyocera['part_number'], checked['specifications']),
]
all_specs = [row for _, _, _, specs in sets for row in specs.values()]
totals = {state: sum(row['verification_status'] == state for row in all_specs) for state in states}
assert totals == {'verified': 6, 'failed': 5, 'unverifiable': 1}
assert run['pdf_validation'] == {kyocera['part_number']: checked}
assert run['totals'] == {'parts_attempted_including_vishay': 3, 'parts_with_response': len(sets),
                         'specifications_reviewed': len(all_specs), 'verification_counts': totals}
assert retry['integrated_totals'] == run['totals']
print(run['totals'])

{'parts_attempted_including_vishay': 3, 'parts_with_response': 2, 'specifications_reviewed': 12, 'verification_counts': {'verified': 6, 'failed': 5, 'unverifiable': 1}}


## 기존 DL 모델 회귀 검사

In [4]:
torch.set_num_threads(min(4,os.cpu_count() or 1))
cache=ROOT/'.venv/model_cache'
pr,mo,de,_=load_table_detector(cache_dir=cache,local_files_only=True)
detector=(pr,mo,de)
pr,mo,de,_=load_structure_recognizer(cache_dir=cache,local_files_only=True)
recognizer=(pr,mo,de)
image,meta,regions,_=detect_page_regions(paths['vishay_t55'],5,detector)
prior=bundle['page_sources']['vishay_t55:p5']
assert len(regions)==len(prior['table_regions'])==1
for region,saved in zip(regions,prior['table_regions']):
    assert np.allclose(region['expanded_bbox_pdf'],saved['expanded_bbox_pdf'],atol=.05,rtol=0)
    assert abs(region['confidence']-saved['confidence'])<1e-6
    current=recognize_region_structure(image,meta,region,recognizer)['objects']
    previous=prior['row_column_candidates'][0]['objects']
    assert len(current)==len(previous)
    assert all(a['label']==b['label'] and abs(a['confidence']-b['confidence'])<1e-6 and
        np.allclose(a['bbox_pdf'],b['bbox_pdf'],atol=.05,rtol=0) for a,b in zip(current,previous))
print('기존 탐지·구조 좌표와 점수 일치함')

Loading weights:   0%|          | 0/367 [00:00<?, ?it/s]

[transformers] TableTransformerForObjectDetection LOAD REPORT from: microsoft/table-transformer-detection
Key                                                                         | Status     |  | 
----------------------------------------------------------------------------+------------+--+-
model.backbone.conv_encoder.model.layer4.0.downsample.1.num_batches_tracked | UNEXPECTED |  | 
model.backbone.conv_encoder.model.layer3.0.downsample.1.num_batches_tracked | UNEXPECTED |  | 
model.backbone.conv_encoder.model.layer2.0.downsample.1.num_batches_tracked | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/367 [00:00<?, ?it/s]

[transformers] TableTransformerForObjectDetection LOAD REPORT from: microsoft/table-transformer-structure-recognition
Key                                                                         | Status     |  | 
----------------------------------------------------------------------------+------------+--+-
model.backbone.conv_encoder.model.layer4.0.downsample.1.num_batches_tracked | UNEXPECTED |  | 
model.backbone.conv_encoder.model.layer3.0.downsample.1.num_batches_tracked | UNEXPECTED |  | 
model.backbone.conv_encoder.model.layer2.0.downsample.1.num_batches_tracked | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


기존 탐지·구조 좌표와 점수 일치함


## 제조사별 규격과 오류 정리

In [5]:
summary_rows = []
errors = pd.read_csv(RESULT_ROOT / errors_rel, dtype=str, keep_default_na=False)
historical = errors.loc[errors.category == 'historical_api'].copy()
historical['manufacturer'] = 'Vishay'
issue_labels = {
    'source_type_mismatch': '근거 유형 불일치',
    'specification_schema_error': '스키마 검사 실패',
    'missing_unit': '필수 단위 누락',
    'unit_unverifiable': '원문 서지 단위 불명확',
    'temperature_range_not_in_primary_evidence': '온도 범위 근거 부족',
    'invalid_unit_value_or_range': '온도 범위 형식 오류',
    'series_temperature_not_target_specific': '대상 P/N 온도 상한 불일치',
    'esr_ambient_temperature_missing': 'ESR 기준 온도 누락',
    'operating_temperature_range_missing': '동작 온도 범위 누락',
    'no_confirmed_value_to_check': '명시된 P/N별 수치 없음',
}
errors_new = []
for maker, doc_id, pn, specs in sets:
    for field, row in specs.items():
        raw = row['raw']
        summary_rows.append(dict(manufacturer=maker, document_id=doc_id, part_number=pn, specification=field,
            extraction_status=raw['status'], verification_status=row['verification_status'], value=raw['value'],
            unit=raw['unit'], condition=raw['condition'], page=raw['page'], scope=raw['scope'],
            issue_types=';'.join(row['issues'])))
        descriptions = {detail['code']: detail['message'] for detail in row.get('issue_details', [])}
        for issue in row['issues']:
            original = {'unit': raw['unit']} if issue in ('missing_unit', 'unit_unverifiable') else raw
            errors_new.append(dict(category='specification', manufacturer=maker, document_id=doc_id,
                part_number=pn, specification=field, error_type=issue,
                raw_field=json.dumps(original, ensure_ascii=False), pdf_check=row['pdf_review']['finding'],
                reason=descriptions.get(issue, issue_labels[issue]),
                unverifiable_reason=row['pdf_review']['finding'] if row['verification_status']=='unverifiable' else '',
                follow_up_needed=True))
errors_new.append(dict(category='current_api', manufacturer='Panasonic', document_id=panasonic['document_id'],
    part_number=panasonic['part_number'], specification=None, error_type='http_503', raw_field=None,
    pdf_check=None, reason=panasonic['error_message'], unverifiable_reason=None, follow_up_needed=True))
errors_new.append(dict(category='panasonic_retry_api', manufacturer='Panasonic',
    document_id=retry_record['document_id'], part_number=retry_record['part_number'],
    specification=None, error_type='http_503', raw_field=None, pdf_check=None,
    reason=retry_record['error_message'], unverifiable_reason=None, follow_up_needed=True))
errors_out = pd.concat([pd.DataFrame(errors_new), historical], ignore_index=True)
assert sum(errors_out['category'] == 'panasonic_retry_api') == 1
summary = pd.DataFrame(summary_rows)
assert len(summary) == 12 and not summary.duplicated(['document_id','part_number','specification']).any()
assert len(historical) == 2 and sum(errors_out['category'] == 'current_api') == 1
display(summary)

,manufacturer,document_id,part_number,specification,extraction_status,verification_status,value,unit,condition,page,scope,issue_types
0,Vishay,vishay_t55,T55A226M2R5C0200,rated_voltage,found,failed,2.5,VDC,NaN,5.0,series,source_type_mismatch
1,Vishay,vishay_t55,T55A226M2R5C0200,surge_voltage,found,failed,3.2,NaN,"105 °C, 1000 successive test cycles in series ...",15.0,series,specification_schema_error;missing_unit;unit_u...
2,Vishay,vishay_t55,T55A226M2R5C0200,capacitance,found,verified,22,μF,NaN,5.0,pn,
3,Vishay,vishay_t55,T55A226M2R5C0200,tolerance,found,verified,± 20,%,NaN,1.0,series,
4,Vishay,vishay_t55,T55A226M2R5C0200,esr,found,verified,200,mΩ,+ 25 °C 100 kHz,5.0,pn,
5,Vishay,vishay_t55,T55A226M2R5C0200,operating_temperature,found,failed,-55 °C to +105 °C / +125 °C,°C,"above 105 °C, additional voltage derating is r...",1.0,series,temperature_range_not_in_primary_evidence;inva...
6,KYOCERA AVX,kyocera_avx_tcj,TCJU337M016R0050E,rated_voltage,found,verified,16,V,NaN,6.0,pn,
7,KYOCERA AVX,kyocera_avx_tcj,TCJU337M016R0050E,surge_voltage,not_found,unverifiable,NaN,NaN,NaN,NaN,NaN,no_confirmed_value_to_check
8,KYOCERA AVX,kyocera_avx_tcj,TCJU337M016R0050E,capacitance,found,verified,330,μF,NaN,6.0,pn,
9,KYOCERA AVX,kyocera_avx_tcj,TCJU337M016R0050E,tolerance,found,verified,±20,%,NaN,1.0,series,


## 추출·검증 상태와 API 오류 시각화

In [6]:
font = Path('C:/Windows/Fonts/malgun.ttf')
font_manager.fontManager.addfont(font)
plt.rcParams.update({'font.family': font_manager.FontProperties(fname=font).get_name(),
                     'axes.unicode_minus': False})
makers = ['Vishay', 'KYOCERA AVX', 'Panasonic']
field_labels = ['정격전압', '서지전압', '정전용량', '허용오차', 'ESR', '동작 온도']
verify_labels = ['검증', '실패', '확정 불가']
verify_colors = dict(zip(states, ('#16745f', '#ad3732', '#865a14')))
fig, (left, right) = plt.subplots(1, 2, figsize=(11.2, 4.8), layout='constrained',
                                 gridspec_kw={'width_ratios': [1.1, 1.5]})
fig.suptitle('7단계 규격 검증 · 호출 P/N 3개 · 응답 2개 · 검증된 규격 6개', fontsize=15)
bottom = np.zeros(len(makers), dtype=int)
for state, color in zip(extraction_states, ('#365e8c', '#84641c', '#6b5ca1', '#b0443f')):
    counts = np.array([sum(row['raw']['status'] == state for row in specs.values()) for _, _, _, specs in sets] + [0])
    bars = left.bar(makers, counts, bottom=bottom, color=color, label=state, width=.58)
    left.bar_label(bars, labels=[str(n) if n else '' for n in counts], label_type='center', color='white')
    bottom += counts
left.text(2, .35, 'HTTP 503', ha='center', color='#9c2828')
left.set(title='Gemini 추출 상태', ylabel='응답 규격 수', ylim=(0, 7))
left.legend(loc='upper center', bbox_to_anchor=(.5, -.09), ncols=2, frameon=False)
left.spines[['top','right']].set_visible(False)
for i, (_, _, _, specs) in enumerate(sets):
    for j, row in enumerate(specs.values()):
        state = row['verification_status']
        right.text(i, j, verify_labels[states.index(state)], ha='center', va='center', color='white',
                   bbox=dict(boxstyle='round,pad=.34', fc=verify_colors[state], ec='none'))
right.text(2, 2.5, '미평가 6개\n재호출 HTTP 503', ha='center', va='center', color='#9c2828')
right.set(title='규격별 최종 검증', xticks=range(3), xticklabels=makers,
          yticks=range(6), yticklabels=field_labels, xlim=(-.5, 2.5), ylim=(5.6, -.6))
right.xaxis.tick_top()
right.tick_params(length=0, pad=8)
right.spines[:].set_visible(False)
right.set_title('규격별 최종 검증', pad=14)
fig.savefig(RESULT_ROOT / status_rel, dpi=155)
plt.close(fig)

issue_counts = [Counter(issue for row in specs.values() for issue in row['issues']) for _, _, _, specs in sets]
issue_order = list(dict.fromkeys(issue for counts in issue_counts for issue in counts))
fig, (left, right) = plt.subplots(1, 2, figsize=(11, 5.2), layout='constrained',
                                 gridspec_kw={'width_ratios': [1.7, 1]})
offset = np.zeros(len(issue_order), dtype=int)
for maker, counts, color in zip(makers, issue_counts, ('#385f8d', '#805227')):
    values = [counts[issue] for issue in issue_order]
    bars = left.barh([issue_labels[issue] for issue in issue_order], values, left=offset, color=color, label=maker)
    left.bar_label(bars, labels=[str(n) if n else '' for n in values], label_type='center', color='white')
    offset += values
left.invert_yaxis()
left.set(title='규격 오류 및 확인 불가 사유', xlabel='사유 수', xlim=(0, max(offset) + .4), xticks=range(max(offset) + 1))
left.legend(loc='upper center', bbox_to_anchor=(.5, -.1), ncols=2, frameon=False)
api_rows = errors_out.loc[errors_out.category.isin(['current_api', 'historical_api', 'panasonic_retry_api'])]
api_counts = [sum((api_rows.manufacturer == maker) & (api_rows.error_type == 'http_503')) for maker in makers]
bars = right.barh(['Vishay (이전)', 'KYOCERA AVX', 'Panasonic (이전+재호출)'], api_counts,
                  color=['#8993a0', '#8993a0', '#ad3732'])
right.bar_label(bars, padding=4)
right.invert_yaxis()
right.set(title='별도 집계 · API HTTP 503', xlabel='요청 수', xlim=(0, max(api_counts) + .5), xticks=range(max(api_counts) + 1))
for axis in (left, right):
    axis.spines[['top','right']].set_visible(False)
fig.savefig(RESULT_ROOT / error_chart_rel, dpi=155)
plt.close(fig)

## 결과 저장 및 재읽기

- 원본 응답과 오류 이력 보존함
- Ground Truth 비교 정확도는 산출하지 않음

In [7]:
result = {**history, 'remaining_pn_run': run, 'panasonic_retry_run': retry}
(RESULT_ROOT / json_rel).write_text(json.dumps(result, ensure_ascii=False, indent=2)+'\n', encoding='utf-8')
summary.to_csv(RESULT_ROOT / summary_rel, index=False, encoding='utf-8-sig')
errors_out.to_csv(RESULT_ROOT / errors_rel, index=False, encoding='utf-8-sig')

readback = json.loads((RESULT_ROOT / json_rel).read_bytes())
assert readback.pop('remaining_pn_run')['totals']['verification_counts'] == totals
assert readback.pop('panasonic_retry_run') == retry
assert readback == history
for relative, frame in ((summary_rel, summary), (errors_rel, errors_out)):
    pd.testing.assert_frame_equal(pd.read_csv(RESULT_ROOT / relative, dtype=str, keep_default_na=False),
                                  frame.fillna('').astype(str))
for relative in (status_rel, error_chart_rel):
    with Image.open(RESULT_ROOT / relative) as plot:
        assert plot.format == 'PNG'
        plot.verify()
assert all(p.suffix.lower() == '.png' for p in (ROOT / 'outputs').rglob('*') if p.is_file())
print('JSON·CSV 재읽기 및 PNG 형식 검증함. 기존 Vishay 기록 동일함.')

JSON·CSV 재읽기 및 PNG 형식 검증함. 기존 Vishay 기록 동일함.


C:\Users\human-20\AppData\Local\Temp\ipykernel_15800\1303897836.py:12: Pandas4Warning: 'str' is not supported as a fill value for float64 dtype. In a future version, calling fillna with an incompatible value will raise. Explicitly cast to a common dtype before filling.
  frame.fillna('').astype(str))
